# M0 gate — does Qwen-Image-Edit-2509 RENOVATE like Gemini?

The single question this notebook answers: run the open, Apache-2.0 editor
**Qwen-Image-Edit-2509** at its **best quality** (speed irrelevant here) on a real
weathered-fence photo — does it *regenerate the wood* (fresh planks, clean grain, no
weathering) the way Gemini did, or does it just tint?

**Judge only:** fresh & uniform wood · weathering/algae/peeling gone · fence structure kept.
**Ignore the colour** — the production color-lock replaces it with the exact swatch later.

**Runtime → GPU.** A100/L4 ideal; T4 (16GB) → set `QUANT_4BIT = True`. First run downloads ~40 GB.
If quality passes here, we build the ~30 s async Cloud Run version (INT4 + Lightning) + finisher.

In [ ]:
!pip -q install -U "diffusers>=0.35" "transformers>=4.55" accelerate safetensors sentencepiece bitsandbytes matplotlib
# Pillow 12 removed PIL._typing._Ink -> breaks the diffusers Qwen import. Pin to 11.x LAST, then RESTART runtime.
!pip -q install "pillow<12"

In [ ]:
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name}   {p.total_memory/1e9:.0f} GB')
if p.total_memory < 20e9: print('  <20GB -> set QUANT_4BIT=True below')

In [ ]:
# ---- load Qwen-Image-Edit-2509 at best quality ----
import torch
QUANT_4BIT = False   # True only on a 16GB T4
FULL_GPU   = torch.cuda.get_device_properties(0).total_memory > 34e9
try:
    from diffusers import QwenImageEditPlusPipeline as QPipe   # 2509 = 'Plus'
except Exception:
    from diffusers import QwenImageEditPipeline as QPipe
IS_PLUS = QPipe.__name__ == 'QwenImageEditPlusPipeline'
kw = dict(torch_dtype=torch.bfloat16)
if QUANT_4BIT:
    from diffusers import PipelineQuantizationConfig
    kw['quantization_config'] = PipelineQuantizationConfig(
        quant_backend='bitsandbytes_4bit',
        quant_kwargs=dict(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16),
        components_to_quantize=['transformer'])
pipe = QPipe.from_pretrained('Qwen/Qwen-Image-Edit-2509', **kw)
pipe.to('cuda') if (FULL_GPU and not QUANT_4BIT) else pipe.enable_model_cpu_offload()
print(f'ready (Plus={IS_PLUS}, full={FULL_GPU}, 4bit={QUANT_4BIT})')

In [ ]:
# ---- upload 1-3 weathered-fence photos ----
import io
from PIL import Image
from google.colab import files
images = {n: Image.open(io.BytesIO(b)).convert('RGB') for n, b in files.upload().items()}
print('loaded:', list(images))

In [ ]:
# ---- RENOVATE (best quality: 40 steps) ----
import torch, matplotlib.pyplot as plt

# Renovation prompt: emphasise REMOVING weathering (renovate, not tint). Colour is a hint only.
PROMPT = ('Restain this wooden privacy fence so it looks freshly and evenly re-stained with fresh, '
  'clean, brand-new {tone} wood. Regenerate the wood surface as newly sanded lumber with fine '
  'natural vertical grain and one uniform even tone across all boards. REMOVE all grey weathering, '
  'water stains, green algae, mildew and peeling paint. Keep the EXACT same fence — same planks, '
  'boards, gaps, rails, posts, dog-ear tops and perspective — and keep every branch, leaf, the '
  'ground and background identical. Photorealistic, sharp, high detail.')
NEGATIVE = ('weathered, faded, grey, peeling, flaking, cracked, old, worn, dirty, mildew, algae, '
  'water stains, blotchy, patchy, uneven, different fence, missing planks, cartoon, blurry, low quality')
TONE, STEPS, TRUE_CFG = 'warm reddish cedar brown', 40, 4.0

def renovate(img, tone=TONE, seed=0):
    g = torch.Generator(device='cpu').manual_seed(seed)
    im = [img] if IS_PLUS else img
    return pipe(image=im, prompt=PROMPT.format(tone=tone), negative_prompt=NEGATIVE,
                num_inference_steps=STEPS, true_cfg_scale=TRUE_CFG, generator=g).images[0]

for name, img in images.items():
    out = renovate(img); out.save('renovated_' + name)
    f, ax = plt.subplots(1, 2, figsize=(16, 8))
    ax[0].imshow(img); ax[0].set_title('original (weathered)'); ax[0].axis('off')
    ax[1].imshow(out); ax[1].set_title('Qwen renovated'); ax[1].axis('off')
    plt.tight_layout(); plt.show()

### The gate
If the right panel shows **fresh, uniform, renovated wood with the weathering gone** and the
**same fence structure** → PASS: send me a couple of these and I build the ~30 s async Cloud Run
`/render` (Qwen INT4 + Lightning LoRA) + the CPU finisher (exact-swatch color-lock + mask composite).

If it still looks weathered or mangles the structure → tell me; we adjust the prompt / steps before
committing to the deployment. Knobs: `STEPS` 30–50, `TRUE_CFG` 3.0–5.0, reword `PROMPT`.